# 13.1 视觉语言 (Vision-Language)

视觉语言模型将视觉信息与语言理解结合，是当前多模态AI最重要的方向。从CLIP到GPT-4V，视觉语言模型正在改变人机交互方式。

## 核心架构演进

| 阶段 | 代表模型 | 架构 | 特点 |
|------|----------|------|------|
| 双编码器 | CLIP | 独立ViT+Text Encoder | 对比学习对齐 |
| 编码器-解码器 | Flamingo | ViT+交叉注意力+LM | 少样本能力 |
| 仅解码器 | LLaVA | ViT+投影+LLM | 指令微调 |
| 原生多模态 | GPT-4V | 统一Transformer | 端到端训练 |

## 1. 视觉编码器

视觉编码器将图像转换为向量表示，是VLM的基础组件。

### ViT（Vision Transformer）
- 将图像切分为固定大小的patch（如14×14）
- 每个patch线性投影为token
- 添加位置编码后输入Transformer
- CLIP ViT是当前最常用的视觉编码器

### 关键设计选择
- **Patch大小**：14×14（高分辨率）vs 16×16（标准）
- **分辨率**：224×224（标准）vs 336×336（高分辨率）vs 动态分辨率
- **预训练**：CLIP对比学习 vs MAE自监督 vs DINO自蒸馏

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class VisionTransformer(nn.Module):
    def __init__(self, img_size=16, patch_size=4, in_channels=3, d_model=64,
                 n_heads=2, n_layers=2, mlp_ratio=4):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        n_patches = (img_size // patch_size) ** 2
        patch_dim = in_channels * patch_size * patch_size

        self.patch_embed = nn.Linear(patch_dim, d_model)
        self.pos_embed = nn.Parameter(torch.randn(1, n_patches, d_model) * 0.02)
        self.cls_token = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.norm = nn.LayerNorm(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model, n_heads, d_model * mlp_ratio, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, n_layers)

    def forward(self, img):
        B, C, H, W = img.shape
        p = self.patch_size
        patches = img.reshape(B, C, H // p, p, W // p, p)
        patches = patches.permute(0, 2, 4, 1, 3, 5).reshape(B, -1, C * p * p)
        tokens = self.patch_embed(patches) + self.pos_embed
        cls_tokens = self.cls_token.expand(B, -1, -1)
        tokens = torch.cat([cls_tokens, tokens], dim=1)
        tokens = self.encoder(tokens)
        tokens = self.norm(tokens)
        return tokens[:, 1:], tokens[:, 0]

class CLIPVisionEncoder(nn.Module):
    def __init__(self, d_vision=64, d_shared=32, temperature=0.07):
        super().__init__()
        self.vit = VisionTransformer(d_model=d_vision)
        self.proj = nn.Linear(d_vision, d_shared)
        self.logit_scale = nn.Parameter(torch.ones(1) * math.log(1 / temperature))

    def encode_image(self, img):
        _, cls = self.vit(img)
        return F.normalize(self.proj(cls), dim=-1)

    def contrastive_loss(self, img_emb, txt_emb):
        scale = self.logit_scale.exp().clamp(max=100)
        logits = scale * img_emb @ txt_emb.T
        labels = torch.arange(logits.shape[0], device=logits.device)
        loss_i2t = F.cross_entropy(logits, labels)
        loss_t2i = F.cross_entropy(logits.T, labels)
        return (loss_i2t + loss_t2i) / 2

vit = VisionTransformer()
img = torch.randn(2, 3, 16, 16)
patch_tokens, cls_token = vit(img)

print('=== Vision Transformer ===')
print(f'Input image: {img.shape}')
print(f'Patch tokens: {patch_tokens.shape} (for dense features)')
print(f'CLS token: {cls_token.shape} (for global features)')

clip_enc = CLIPVisionEncoder()
img_emb = clip_enc.encode_image(img)
txt_emb = F.normalize(torch.randn(2, 32), dim=-1)
clip_loss = clip_enc.contrastive_loss(img_emb, txt_emb)
print(f'\nCLIP image embedding: {img_emb.shape}')
print(f'Contrastive loss: {clip_loss.item():.4f}')
print(f'\nKey: CLS token captures global semantics, patch tokens preserve spatial info.')

## 2. 视觉语言模型（VLM）

VLM将视觉特征与语言模型结合，实现图像理解和视觉问答。

### 三种主流架构

**1. 投影式（LLaVA风格）**
- ViT → 线性/MLP投影 → LLM
- 简单高效，最流行的方案
- 投影层将视觉token映射到语言空间

**2. 交叉注意力式（Flamingo风格）**
- ViT → 交叉注意力 → LLM
- 视觉token通过交叉注意力注入LLM各层
- 更强的视觉理解能力，但计算量更大

**3. 原生多模态（GPT-4V风格）**
- 视觉和文本token在统一Transformer中处理
- 端到端训练，无投影层
- 最强但训练成本最高

In [ ]:
class LLaVAStyleVLM(nn.Module):
    def __init__(self, d_vision=64, d_language=128, vocab_size=500, n_vis_tokens=16):
        super().__init__()
        self.vision_encoder = VisionTransformer(d_model=d_vision)
        self.vision_proj = nn.Sequential(
            nn.Linear(d_vision, d_language),
            nn.GELU(),
            nn.Linear(d_language, d_language)
        )
        self.text_embed = nn.Embedding(vocab_size, d_language)
        self.lm = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_language, 2, d_language * 4, batch_first=True), 2
        )
        self.head = nn.Linear(d_language, vocab_size)
        self.n_vis_tokens = n_vis_tokens

    def forward(self, image, text_tokens):
        vis_features, _ = self.vision_encoder(image)
        vis_tokens = self.vision_proj(vis_features)
        txt_tokens = self.text_embed(text_tokens)
        combined = torch.cat([vis_tokens, txt_tokens], dim=1)
        h = self.lm(combined)
        text_output = h[:, vis_tokens.shape[1]:]
        return self.head(text_output)

class FlamingoStyleVLM(nn.Module):
    def __init__(self, d_vision=64, d_language=128, vocab_size=500, n_layers=2):
        super().__init__()
        self.vision_encoder = VisionTransformer(d_model=d_vision)
        self.vision_proj = nn.Linear(d_vision, d_language)
        self.text_embed = nn.Embedding(vocab_size, d_language)
        self.cross_attn_layers = nn.ModuleList([
            nn.MultiheadAttention(d_language, 2, batch_first=True)
            for _ in range(n_layers)
        ])
        self.lm_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(d_language, 2, d_language * 4, batch_first=True)
            for _ in range(n_layers)
        ])
        self.head = nn.Linear(d_language, vocab_size)

    def forward(self, image, text_tokens):
        vis_features, _ = self.vision_encoder(image)
        vis_tokens = self.vision_proj(vis_features)
        txt_tokens = self.text_embed(text_tokens)

        h = txt_tokens
        for cross_attn, lm_layer in zip(self.cross_attn_layers, self.lm_layers):
            h, _ = cross_attn(h, vis_tokens, vis_tokens)
            h = lm_layer(h)

        return self.head(h)

img = torch.randn(2, 3, 16, 16)
text = torch.randint(0, 500, (2, 10))

llava = LLaVAStyleVLM()
flamingo = FlamingoStyleVLM()

llava_out = llava(img, text)
flamingo_out = flamingo(img, text)

print('=== VLM Architecture Comparison ===')
print(f'\nLLaVA-style (Projection):')
print(f'  Image: {img.shape} -> Vision tokens -> Project -> Concat with text -> LLM')
print(f'  Output: {llava_out.shape}')
llava_params = sum(p.numel() for p in llava.parameters())
print(f'  Parameters: {llava_params:,}')

print(f'\nFlamingo-style (Cross-Attention):')
print(f'  Image: {img.shape} -> Vision tokens -> Cross-attention at each LM layer')
print(f'  Output: {flamingo_out.shape}')
flamingo_params = sum(p.numel() for p in flamingo.parameters())
print(f'  Parameters: {flamingo_params:,}')

print(f'\nKey: LLaVA is simpler and faster. Flamingo has deeper vision-language fusion.')
print(f'LLaVA is preferred for most production use cases.')

## 3. 视觉指令微调

视觉指令微调使VLM能遵循视觉相关指令，是VLM从"看图说话"到"看图思考"的关键步骤。

### LLaVA训练流程
1. **阶段1：预训练对齐**
   - 冻结ViT和LLM，只训练投影层
   - 使用图像-标题对齐数据
   - 目标：让投影层学会将视觉特征映射到语言空间

2. **阶段2：视觉指令微调**
   - 解冻LLM（可选冻结ViT）
   - 使用图像-指令-回复三元组
   - 目标：学会遵循视觉指令

### 数据构建
- **LLaVA-Instruct**：用GPT-4基于图像描述生成指令数据
- **ShareGPT4V**：用GPT-4V生成高质量视觉指令数据
- **人工标注**：质量最高但成本大

In [ ]:
class VisualInstructionTuner:
    def __init__(self, vlm, lr_pretrain=1e-3, lr_finetune=1e-5):
        self.vlm = vlm
        self.lr_pretrain = lr_pretrain
        self.lr_finetune = lr_finetune

    def stage1_pretrain_alignment(self, images, captions, n_epochs=10):
        for name, param in self.vlm.named_parameters():
            if 'vision_proj' not in name:
                param.requires_grad = False
            else:
                param.requires_grad = True

        optimizer = torch.optim.AdamW(
            [p for p in self.vlm.parameters() if p.requires_grad],
            lr=self.lr_pretrain
        )

        print('Stage 1: Pretrain alignment (projection only)')
        trainable = sum(p.numel() for p in self.vlm.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.vlm.parameters())
        print(f'  Trainable: {trainable:,} / {total:,} ({trainable/total:.1%})')

        for epoch in range(n_epochs):
            logits = self.vlm(images, captions[:, :-1])
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                                   captions[:, 1:].reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if (epoch + 1) % 3 == 0:
                print(f'  Epoch {epoch+1}: loss={loss.item():.4f}')

    def stage2_instruction_finetune(self, images, instructions, responses, n_epochs=10):
        for name, param in self.vlm.named_parameters():
            if 'vision_encoder' in name:
                param.requires_grad = False
            else:
                param.requires_grad = True

        optimizer = torch.optim.AdamW(
            [p for p in self.vlm.parameters() if p.requires_grad],
            lr=self.lr_finetune
        )

        print(f'\nStage 2: Instruction fine-tuning (unfreeze LLM)')
        trainable = sum(p.numel() for p in self.vlm.parameters() if p.requires_grad)
        total = sum(p.numel() for p in self.vlm.parameters())
        print(f'  Trainable: {trainable:,} / {total:,} ({trainable/total:.1%})')

        for epoch in range(n_epochs):
            logits = self.vlm(images, instructions)
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                                   responses.reshape(-1))
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(self.vlm.parameters(), 1.0)
            optimizer.step()
            if (epoch + 1) % 3 == 0:
                print(f'  Epoch {epoch+1}: loss={loss.item():.4f}')

vlm = LLaVAStyleVLM(d_vision=64, d_language=128, vocab_size=500)
tuner = VisualInstructionTuner(vlm)

images = torch.randn(4, 3, 16, 16)
captions = torch.randint(0, 500, (4, 10))
instructions = torch.randint(0, 500, (4, 8))
responses = torch.randint(0, 500, (4, 8))

print('=== Visual Instruction Tuning ===')
tuner.stage1_pretrain_alignment(images, captions, n_epochs=9)
tuner.stage2_instruction_finetune(images, instructions, responses, n_epochs=9)

print(f'\nKey: Stage 1 aligns vision-language spaces (fast, projection only).')
print(f'Stage 2 teaches instruction following (slower, unfreezes LLM).')

## 4. 高分辨率与动态分辨率

标准VLM使用固定低分辨率（224×224），会丢失细节信息。高分辨率和动态分辨率是当前VLM的重要改进方向。

### 方案对比
| 方案 | 分辨率 | 计算量 | 细节保留 | 代表模型 |
|------|--------|--------|----------|----------|
| 固定低分辨率 | 224×224 | 低 | 差 | LLaVA-1.5 |
| 固定高分辨率 | 336×336 | 中 | 中 | LLaVA-1.5-HD |
| 动态分辨率 | 任意 | 自适应 | 好 | Qwen-VL |
| 切片+全局 | 多切片 | 高 | 最好 | LLaVA-NeXT |

### 切片策略（AnyRes）
将高分辨率图像切分为多个子图，分别编码后合并：
1. 全局视图：缩放到标准分辨率，获取全局语义
2. 局部切片：高分辨率切片，获取细节信息
3. 特征合并：将全局和局部特征拼接后输入LLM

In [ ]:
class AnyResVisionEncoder(nn.Module):
    def __init__(self, base_encoder, d_vision=64, d_language=128,
                 max_patches=4, patch_size=4):
        super().__init__()
        self.base_encoder = base_encoder
        self.max_patches = max_patches
        self.patch_proj = nn.Linear(d_vision * 2, d_vision)
        self.vision_proj = nn.Linear(d_vision, d_language)

    def split_into_patches(self, img, grid_size=2):
        B, C, H, W = img.shape
        h, w = H // grid_size, W // grid_size
        patches = img.reshape(B, C, grid_size, h, grid_size, w)
        patches = patches.permute(0, 2, 4, 1, 3, 5).reshape(B * grid_size * grid_size, C, h, w)
        return patches

    def forward(self, img):
        B = img.shape[0]
        global_features, global_cls = self.base_encoder(img)
        global_cls = global_cls.unsqueeze(1)

        patches = self.split_into_patches(img, grid_size=2)
        patch_features, _ = self.base_encoder(patches)
        n_patch_tokens = patch_features.shape[1]
        patch_features = patch_features.reshape(B, 4, n_patch_tokens, -1)
        patch_features = patch_features.reshape(B, 4 * n_patch_tokens, -1)

        global_expanded = global_cls.expand(-1, patch_features.shape[1], -1)
        combined = torch.cat([global_expanded, patch_features], dim=-1)
        combined = self.patch_proj(combined)
        return self.vision_proj(combined)

base_vit = VisionTransformer(d_model=64)
anyres = AnyResVisionEncoder(base_vit, d_vision=64, d_language=128)

img = torch.randn(2, 3, 16, 16)
vis_tokens = anyres(img)

print('=== AnyRes (Dynamic Resolution) ===')
print(f'Input image: {img.shape}')
print(f'Global + Patch features -> Vision tokens: {vis_tokens.shape}')
print(f'\nKey: AnyRes encodes both global view and local patches.')
print(f'This preserves fine-grained details for tasks like OCR and document understanding.')

print(f'\n=== VLM Industrial Best Practices ===')
print(f'1. Use CLIP ViT-L/14 as vision encoder (best quality-efficiency tradeoff)')
print(f'2. 2-layer MLP projection works better than linear projection')
print(f'3. Stage 1: 595K image-caption pairs, Stage 2: 665K instruction pairs')
print(f'4. Freeze ViT during instruction tuning to preserve visual features')
print(f'5. Use AnyRes for high-resolution tasks (OCR, document, medical imaging)')
print(f'6. Mix text-only data during VLM training to preserve language ability')